In [ ]:
# did_utils.py

import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt


# ============================================================
# Helper
# ============================================================

def unique_list(seq):
    """Remove duplicates while preserving order."""
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out


# ============================================================
# Build matched panel
# ============================================================

def build_matched_panel(matches, month_result, outcome_cols=None):
    """
    Build matched panel for matched DiD.

    Required columns in matches:
        treated_id
        control_id
        adoption_month

    Required columns in month_result:
        aID
        TIDPUNKT
        outcome columns

    Parameters
    ----------
    matches : pd.DataFrame
        Matching result.
    month_result : pd.DataFrame
        Household-month panel data.
    outcome_cols : dict, optional
        Dictionary of outcome names and column names, e.g.
        {"H": "high_peak", "L": "low_peak", "T": "total_consumption"}.
    """

    matches = matches.copy()
    month_result = month_result.copy()

    # Adoption cohort
    matches["cohort"] = pd.to_datetime(matches["adoption_month"]).dt.to_period("M")

    # Treated households
    treated_map = matches[["treated_id", "cohort"]].drop_duplicates().copy()
    treated_map.columns = ["aID", "cohort"]
    treated_map["treatment"] = 1
    treated_map["match_weight"] = 1.0

    # Control households
    # Controls are weighted by 1 / number of matches per treated household.
    # If the same control is used multiple times, weights are summed.
    matches["n_controls_for_treated"] = matches.groupby(
        ["treated_id", "cohort"]
    )["control_id"].transform("count")

    control_map = matches[["control_id", "cohort", "n_controls_for_treated"]].copy()
    control_map.columns = ["aID", "cohort", "n_controls_for_treated"]
    control_map["treatment"] = 0
    control_map["match_weight"] = 1.0 / control_map["n_controls_for_treated"]

    control_map = (
        control_map
        .groupby(["aID", "cohort", "treatment"], as_index=False)["match_weight"]
        .sum()
    )

    # Combine treated and controls
    match_map = pd.concat(
        [
            treated_map[["aID", "cohort", "treatment", "match_weight"]],
            control_map[["aID", "cohort", "treatment", "match_weight"]]
        ],
        axis=0,
        ignore_index=True
    )

    # Merge with monthly data
    df = month_result.merge(match_map, on="aID", how="inner")

    # Time variables
    df["TIDPUNKT"] = pd.to_datetime(df["TIDPUNKT"]).dt.to_period("M")
    df["month_id"] = df["TIDPUNKT"].astype(str)
    df["cohort_id"] = df["cohort"].astype(str)

    # Event time
    df["event_time"] = (df["TIDPUNKT"] - df["cohort"]).apply(lambda x: x.n)

    # Post and treatment indicator
    df["post"] = (df["event_time"] >= 0).astype(int)
    df["D"] = df["treatment"] * df["post"]

    # Same control household can appear in different cohorts.
    # Treat each household-cohort combination as a separate matched-panel unit.
    df["entity_id"] = (
        df["aID"].astype(str)
        + "_cohort_"
        + df["cohort_id"].astype(str)
        + "_tr_"
        + df["treatment"].astype(str)
    )

    # Convert outcomes to numeric
    if outcome_cols is not None:
        for col in outcome_cols.values():
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
            else:
                print(f"Warning: outcome column not found: {col}")

    return df


# ============================================================
# Two-way fixed effects residualization
# ============================================================

def weighted_group_mean(frame, cols, group_col, weight_col):
    """
    Compute weighted group means for multiple columns.
    """
    w = frame[weight_col].astype(float)
    tmp = frame[cols].multiply(w, axis=0)
    numerator = tmp.groupby(frame[group_col]).sum()
    denominator = w.groupby(frame[group_col]).sum()
    means = numerator.div(denominator, axis=0)
    return means


def residualize_twfe(
    data,
    cols,
    entity_col="entity_id",
    time_col="month_id",
    weight_col="match_weight",
    keep_cols=None,
    max_iter=100,
    tol=1e-10
):
    """
    Residualize columns with respect to entity and time fixed effects
    using iterative weighted demeaning.

    This avoids explicitly creating thousands of dummy variables.
    """

    if keep_cols is None:
        keep_cols = []

    base_cols = unique_list([entity_col, time_col, weight_col] + keep_cols + cols)

    work = data[base_cols].copy()
    work = work.dropna(subset=cols + [entity_col, time_col, weight_col])

    R = work[cols].astype(float).copy()

    for _ in range(max_iter):
        old_values = R.values.copy()

        # Remove entity fixed effects
        temp = pd.concat(
            [
                work[[entity_col, weight_col]].reset_index(drop=True),
                R.reset_index(drop=True)
            ],
            axis=1
        )

        entity_means = weighted_group_mean(
            temp, cols, entity_col, weight_col
        )

        R -= entity_means.reindex(work[entity_col]).to_numpy()

        # Remove time fixed effects
        temp = pd.concat(
            [
                work[[time_col, weight_col]].reset_index(drop=True),
                R.reset_index(drop=True)
            ],
            axis=1
        )

        time_means = weighted_group_mean(
            temp, cols, time_col, weight_col
        )

        R -= time_means.reindex(work[time_col]).to_numpy()

        diff = np.nanmax(np.abs(R.values - old_values))

        if diff < tol:
            break

    R.index = work.index

    return work, R


# ============================================================
# Basic DiD estimation
# ============================================================

def fit_basic_did(
    df,
    outcome_col,
    outcome_name,
    entity_col="entity_id",
    time_col="month_id",
    weight_col="match_weight",
    cluster_col="entity_id"
):
    """
    Estimate:
        Y_it = alpha_i + lambda_t + beta D_it + e_it

    using weighted two-way fixed effects residualization.
    """

    use_cols = unique_list(
        [outcome_col, "D", entity_col, time_col, weight_col, cluster_col]
    )

    data = df[use_cols].dropna(subset=[outcome_col, "D", entity_col, time_col, weight_col, cluster_col]).copy()

    work, R = residualize_twfe(
        data=data,
        cols=[outcome_col, "D"],
        entity_col=entity_col,
        time_col=time_col,
        weight_col=weight_col,
        keep_cols=[cluster_col]
    )

    y = R[outcome_col]
    X = R[["D"]]
    weights = work[weight_col].astype(float)

    model = sm.WLS(y, X, weights=weights)

    result = model.fit(
        cov_type="cluster",
        cov_kwds={"groups": work[cluster_col]}
    )

    coef = result.params["D"]
    se = result.bse["D"]
    pval = result.pvalues["D"]

    out = {
        "outcome": outcome_name,
        "column": outcome_col,
        "coef": coef,
        "std_error": se,
        "p_value": pval,
        "ci_lower": coef - 1.96 * se,
        "ci_upper": coef + 1.96 * se,
        "n_obs": len(work),
        "n_entities": work[entity_col].nunique()
    }

    return result, out


def run_basic_did_all(df, outcome_cols, cluster_col="aID"):
    """
    Run baseline DiD for all outcomes in outcome_cols.
    """

    results = []
    models = {}

    for outcome_name, outcome_col in outcome_cols.items():
        print(f"Running basic DiD for {outcome_name}: {outcome_col}")

        model, row = fit_basic_did(
            df=df,
            outcome_col=outcome_col,
            outcome_name=outcome_name,
            cluster_col=cluster_col
        )

        models[outcome_name] = model
        results.append(row)

    results_df = pd.DataFrame(results)

    return models, results_df



def run_basic_did_by_cohort(
    df,
    outcome_cols,
    cohorts=None,
    min_treated=30,
    entity_col="entity_id",
    time_col="month_id",
    weight_col="match_weight",
    cluster_col="aID"
):
    """
    Run baseline DiD separately for each adoption cohort and each outcome.

    Estimate within each cohort:
        Y_it = alpha_i + lambda_t + beta D_it + e_it

    Returns
    -------
    cohort_models : dict
        Models indexed by (cohort, outcome).
    cohort_results_df : pd.DataFrame
        ATT estimates by cohort and outcome.
    """

    if cohorts is None:
        cohorts = sorted(df["cohort_id"].dropna().unique())

    cohort_models = {}
    all_results = []

    for cohort in cohorts:
        df_g = df[df["cohort_id"] == cohort].copy()

        n_treated = df_g.loc[df_g["treatment"] == 1, "aID"].nunique()

        if n_treated < min_treated:
            print(f"Skipping cohort {cohort}: only {n_treated} treated households")
            continue

        n_periods = df_g.loc[df_g["D"] == 1, "month_id"].nunique()

        print(f"Running cohort {cohort}, treated households = {n_treated}")

        for outcome_name, outcome_col in outcome_cols.items():
            try:
                model, row = fit_basic_did(
                    df=df_g,
                    outcome_col=outcome_col,
                    outcome_name=outcome_name,
                    entity_col=entity_col,
                    time_col=time_col,
                    weight_col=weight_col,
                    cluster_col=cluster_col
                )

                row["cohort"] = cohort
                row["n_treated"] = n_treated
                row["n_periods"] = n_periods

                cohort_models[(cohort, outcome_name)] = model
                all_results.append(row)

            except Exception as e:
                print(f"Failed cohort {cohort}, outcome {outcome_name}: {e}")

    if len(all_results) == 0:
        return cohort_models, pd.DataFrame()

    cohort_results_df = pd.DataFrame(all_results)

    # Reorder columns
    preferred_cols = [
        "cohort", "outcome", "coef", "std_error", "p_value",
        "ci_lower", "ci_upper", "n_treated", "n_periods",
        "n_obs", "n_entities", "column"
    ]

    existing_cols = [c for c in preferred_cols if c in cohort_results_df.columns]
    other_cols = [c for c in cohort_results_df.columns if c not in existing_cols]

    cohort_results_df = cohort_results_df[existing_cols + other_cols]

    return cohort_models, cohort_results_df


# ============================================================
# Event-study estimation
# ============================================================

def make_event_dummies(
    data,
    event_col="event_time",
    treatment_col="treatment",
    event_window=(-12, 12),
    reference=-1
):
    """
    Create treatment x event-time dummies.
    Reference period is omitted.
    """

    data = data.copy()

    min_k, max_k = event_window
    data = data[(data[event_col] >= min_k) & (data[event_col] <= max_k)].copy()

    event_terms = []

    for k in range(min_k, max_k + 1):
        if k == reference:
            continue

        col = f"event_{k}"
        data[col] = ((data[event_col] == k) & (data[treatment_col] == 1)).astype(int)
        event_terms.append(col)

    return data, event_terms


def fit_event_study(
    df,
    outcome_col,
    outcome_name,
    event_window=(-12, 12),
    reference=-1,
    entity_col="entity_id",
    time_col="month_id",
    weight_col="match_weight",
    cluster_col="entity_id"
):
    """
    Estimate event-study model:

        Y_it = alpha_i + lambda_t
               + sum_{k != -1} beta_k [Treated_i x 1(event_time = k)]
               + e_it
    """

    data, event_terms = make_event_dummies(
        df,
        event_col="event_time",
        treatment_col="treatment",
        event_window=event_window,
        reference=reference
    )

    use_cols = unique_list(
        [outcome_col] + event_terms + [entity_col, time_col, weight_col, cluster_col]
    )

    data = data[use_cols].dropna(subset=[outcome_col, entity_col, time_col, weight_col, cluster_col]).copy()

    # Drop event dummies with no variation
    valid_terms = []
    for col in event_terms:
        if col in data.columns and data[col].sum() > 0 and data[col].var() > 0:
            valid_terms.append(col)

    if len(valid_terms) == 0:
        raise ValueError("No valid event-time dummies. Check event_window and data.")

    cols_to_residualize = [outcome_col] + valid_terms

    work, R = residualize_twfe(
        data=data,
        cols=cols_to_residualize,
        entity_col=entity_col,
        time_col=time_col,
        weight_col=weight_col,
        keep_cols=[cluster_col]
    )

    y = R[outcome_col]
    X = R[valid_terms]
    weights = work[weight_col].astype(float)

    model = sm.WLS(y, X, weights=weights)

    result = model.fit(
        cov_type="cluster",
        cov_kwds={"groups": work[cluster_col]}
    )

    rows = []

    for term in valid_terms:
        k = int(term.replace("event_", ""))
        coef = result.params[term]
        se = result.bse[term]

        rows.append({
            "outcome": outcome_name,
            "event_time": k,
            "coef": coef,
            "std_error": se,
            "p_value": result.pvalues[term],
            "ci_lower": coef - 1.96 * se,
            "ci_upper": coef + 1.96 * se
        })

    event_df = pd.DataFrame(rows).sort_values("event_time")

    return result, event_df


def run_event_study_all(
    df,
    outcome_cols,
    event_window=(-12, 12),
    reference=-1,
    cluster_col="aID"
):
    """
    Run event-study for all outcomes in outcome_cols.
    """

    event_models = {}
    event_results = []

    for outcome_name, outcome_col in outcome_cols.items():
        print(f"Running event-study for {outcome_name}: {outcome_col}")

        model, event_df = fit_event_study(
            df=df,
            outcome_col=outcome_col,
            outcome_name=outcome_name,
            event_window=event_window,
            reference=reference,
            cluster_col=cluster_col
        )

        event_models[outcome_name] = model
        event_results.append(event_df)

    event_results_df = pd.concat(event_results, ignore_index=True)

    return event_models, event_results_df



# ============================================================
# Plot event-study results
# ============================================================

def plot_event_study(event_results_df, outcome_name, title=None, save_path=None):
    """
    Plot event-study coefficients with 95% confidence intervals.
    """

    plot_df = event_results_df[event_results_df["outcome"] == outcome_name].copy()
    plot_df = plot_df.sort_values("event_time")

    if plot_df.empty:
        raise ValueError(f"No event-study results found for outcome: {outcome_name}")

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.axhline(0, linewidth=1)
    ax.axvline(-0.5, linestyle="--", linewidth=1)

    ax.errorbar(
        plot_df["event_time"],
        plot_df["coef"],
        yerr=1.96 * plot_df["std_error"],
        fmt="o",
        capsize=3
    )

    ax.set_xlabel("Event time relative to tariff adoption")
    ax.set_ylabel("Estimated effect")
    ax.set_title(title if title is not None else f"Event-study: {outcome_name}")

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return fig, ax



def run_event_study_by_cohort(
    df,
    outcome_cols,
    cohorts=None,
    event_window=(-12, 12),
    reference=-1,
    min_treated=30,
    cluster_col="aID"
):
    """
    Run event-study separately for each adoption cohort.

    Parameters
    ----------
    df : pd.DataFrame
        Matched panel.
    outcome_cols : dict
        Example: {"H": "high_price_peak", "L": "low_price_peak", "T": "total_consumption"}
    cohorts : list or None
        Cohorts to estimate. If None, all cohorts are used.
    event_window : tuple
        Event-time window, e.g. (-12, 6).
    reference : int
        Reference event time, usually -1.
    min_treated : int
        Minimum number of treated households required in a cohort.

    Returns
    -------
    cohort_event_models : dict
        Models indexed by (cohort, outcome).
    cohort_event_results_df : pd.DataFrame
        Event-study coefficients for each cohort and outcome.
    """

    if cohorts is None:
        cohorts = sorted(df["cohort_id"].dropna().unique())

    cohort_event_models = {}
    all_results = []

    for cohort in cohorts:
        df_g = df[df["cohort_id"] == cohort].copy()

        n_treated = df_g.loc[df_g["treatment"] == 1, "aID"].nunique()

        if n_treated < min_treated:
            print(f"Skipping cohort {cohort}: only {n_treated} treated households")
            continue

        print(f"Running cohort {cohort}, treated households = {n_treated}")

        for outcome_name, outcome_col in outcome_cols.items():
            try:
                model, event_df = fit_event_study(
                    df=df_g,
                    outcome_col=outcome_col,
                    outcome_name=outcome_name,
                    event_window=event_window,
                    reference=reference,
                    cluster_col=cluster_col
                )

                event_df["cohort"] = cohort
                event_df["n_treated"] = n_treated

                cohort_event_models[(cohort, outcome_name)] = model
                all_results.append(event_df)

            except Exception as e:
                print(f"Failed cohort {cohort}, outcome {outcome_name}: {e}")

    if len(all_results) == 0:
        return cohort_event_models, pd.DataFrame()

    cohort_event_results_df = pd.concat(all_results, ignore_index=True)

    return cohort_event_models, cohort_event_results_df



def plot_event_study_by_cohort(
    cohort_event_results_df,
    outcome_name,
    cohorts=None,
    title=None
):
    """
    Plot cohort-specific event-study curves for one outcome.
    """

    plot_df = cohort_event_results_df[
        cohort_event_results_df["outcome"] == outcome_name
    ].copy()

    if cohorts is not None:
        plot_df = plot_df[plot_df["cohort"].isin(cohorts)]

    if plot_df.empty:
        raise ValueError(f"No results found for outcome {outcome_name}")

    plt.figure(figsize=(10, 6))

    plt.axhline(0, linewidth=1)
    plt.axvline(-0.5, linestyle="--", linewidth=1)

    for cohort, group in plot_df.groupby("cohort"):
        group = group.sort_values("event_time")
        plt.plot(
            group["event_time"],
            group["coef"],
            marker="o",
            label=str(cohort)
        )

    plt.xlabel("Event time relative to tariff adoption")
    plt.ylabel("Estimated effect")
    plt.title(title if title is not None else f"Cohort-specific event-study: {outcome_name}")
    plt.legend(title="Cohort", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()